# HW7 — Validation, leakage and uncertainty

## Why this notebook exists

`HW6` produced an uncomfortable result: a zoo of ARIMA, ETS and state-space models beat the
random walk on SPY returns by a wide margin, and *none* of them beat a constant mean. The margin
was real and the conclusion was empty, because the benchmark had been implemented incorrectly.

That is the general failure mode of this field, and it deserves a notebook that attacks it
directly rather than one more model. The vacancy brief asks for validation, leakage control and
backtest uncertainty, and `HW1`–`HW5` contain none of them in a form that could be defended in a
code review. So this notebook does not train a model to make a prediction. It tries hard to
**falsify** results, using the same data and the same models throughout so that the only thing
varying is the validation discipline.

## The five questions

Each is answered by an experiment that keeps the model fixed and changes only the method:

| # | Question | Experiment |
|---|---|---|
| 1 | Does the *split rule* change the answer? | random K-fold vs purged K-fold vs walk-forward |
| 2 | How much does preprocessing leakage inflate the number? | fold-safe scaler vs scaler fitted on train+test |
| 3 | Can a metric be gamed without a causal-feature audit? | one forward-looking feature, added on purpose |
| 4 | How much of a "best model" is just selection noise? | 24 configurations, selected on CV, scored out of sample |
| 5 | How much does the answer depend on *where* the test window sat? | combinatorial purged CV path distribution |

## The standard this notebook holds itself to

Three rules, learned the hard way in `HW6`:

* **Every metric is paired with a zero-parameter benchmark.** Accuracy is meaningless next to
  majority-class accuracy; Sharpe is meaningless next to buy-and-hold; a calibrated interval is
  meaningless if the miscoverage rate is never quoted.
* **Nothing is selected on the final test window.** The last 20% of bars is touched once, by the
  one model the honest procedure chose, and is otherwise used as a measuring instrument.
* **Uncertainty is reported, not implied.** A point estimate of a Sharpe ratio from a few hundred
  bars is a claim about noise. Every headline number here comes with an interval.

Where the honest answer is "no edge", the notebook says so. A negative result, clearly stated, is
worth more to a research team than a positive one that dissolves under inspection.

In [1]:
import json
import os
import sys
import warnings
from itertools import product

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

from mlfin_core import (
    TRADING_DAYS,
    FoldSafePreprocessor,
    annualised_return,
    apply_causal_execution,
    bootstrap_sharpe_ci,
    buy_and_hold_returns,
    cpcv_backtest_paths,
    combinatorial_purged_cv_splits,
    deflated_sharpe_ratio,
    load_ohlcv,
    make_direction_target,
    make_feature_frame,
    min_track_record_length,
    positions_to_returns,
    probabilistic_sharpe_ratio,
    profit_factor,
    purged_kfold_splits,
    sharpe_ratio,
    sortino_ratio,
    walk_forward_splits,
    win_rate,
)
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

print("mlfin_core imported | TRADING_DAYS =", TRADING_DAYS)

mlfin_core imported | TRADING_DAYS = 252


## Configuration

Two choices deserve their own defence, and both are the ones most often got wrong in this
repository's earlier notebooks.

**A deadband on the target.** `make_direction_target` supports a `deadband`: a forward return
inside ±`DEAD_BAND` becomes a third *hold* class rather than being forced into up or down. Without
it the target is "did the price tick up", which is mostly bid-ask bounce and microstructure noise,
and a model can score well on it while having nothing to trade.

**A held-out final window.** The last `OOS_FRACTION` of bars is never used for model selection,
preprocessing statistics, or early stopping. `EMBARGO` bars are additionally dropped around test
blocks, because a label whose window straddles the test boundary has seen the test period.

`REFIT_EVERY` matches `HW6`. Refitting on every bar is cleaner in principle and unaffordable in
practice; refitting weekly and always scoring *before* the next refit keeps the simulation causal.

In [2]:
TICKER = "SPY"
START = "2015-01-01"
HORIZON = 1              # decision-time label horizon, in bars
DEAD_BAND = 0.0005       # +/- 5 bp becomes the "hold" class
EMBARGO = 5              # bars dropped either side of a test block
OOS_FRACTION = 0.20      # final window: touched exactly once
REFIT_EVERY = 21         # walk-forward refit cadence
COST_BPS = 10.0          # round-trip cost assumption
N_SPLITS = 5
SR_TARGET = 1.0          # annualised Sharpe the track record is judged against
RNG_SEED = 0

df, source = load_ohlcv(TICKER, START, allow_synthetic=True)
print(f"{TICKER}: {len(df)} bars | {df.index[0].date()} -> {df.index[-1].date()}")
print(f"data source: {source}")
if "synthetic" in str(source).lower():
    print("WARNING: synthetic fallback in use. Every number below is then a property of the")
    print("generator, not of a market. Conclusions must be read as method demonstrations.")

SPY: 3066 bars | 2015-01-02 -> 2026-10-02
data source: SPY (yfinance: 2015-01-01..2026-10-05)


## Features, target, and the decision-time contract

`make_feature_frame` returns columns that are functions of the close of bar *t* and earlier.
`make_direction_target` returns the label for bar *t*, which is only **observable** at *t + HORIZON*.
That gap is the entire reason purging and embargo exist, and the next cell measures it rather than
asserting it.

The majority-class frequency is recorded here, on the training portion only. It becomes the
accuracy benchmark in every experiment below — the `HW6` lesson, applied to classification.

In [3]:
feats = make_feature_frame(df, n_lags=5, extra_indicators=True)
y = make_direction_target(df["Close"], horizon=HORIZON, deadband=DEAD_BAND, labels=(0, 1, 2))
y = y.rename("y")

data = feats.join(y, how="inner").dropna(how="all")
data = data[data["y"].notna()].copy()
close = df["Close"].reindex(data.index)

n_obs = len(data)
n_oos = int(round(n_obs * OOS_FRACTION))
IS_SLICE = slice(0, n_obs - n_oos)
OOS_SLICE = slice(n_obs - n_oos, n_obs)

X = data.drop(columns=["y"])
y = data["y"]
X_IS, y_IS = X.iloc[IS_SLICE], y.iloc[IS_SLICE]
X_OOS, y_OOS = X.iloc[OOS_SLICE], y.iloc[OOS_SLICE]
close_IS, close_OOS = close.iloc[IS_SLICE], close.iloc[OOS_SLICE]

CLASS_NAMES = {0: "sell", 1: "buy", 2: "hold"}
print(f"features        : {X.shape[1]} columns, {X.shape[0]} rows")
print(f"feature window  : {X.index[0].date()} -> {X.index[-1].date()}")
print(f"IS   (selection): {len(X_IS)} bars  {X_IS.index[0].date()} -> {X_IS.index[-1].date()}")
print(f"OOS  (one touch): {len(X_OOS)} bars  {X_OOS.index[0].date()} -> {X_OOS.index[-1].date()}")

print("\nclass balance")
bal = pd.DataFrame({
    "IS_share": y_IS.value_counts(normalize=True).sort_index(),
    "OOS_share": y_OOS.value_counts(normalize=True).sort_index(),
})
bal.index = [CLASS_NAMES[i] for i in bal.index]
print(bal.round(4).to_string())

# The benchmark. Fitted on IS only, applied to OOS without ever looking at it.
MAJORITY_CLASS = int(y_IS.value_counts().idxmax())
CLIMATOLOGY_ACC = float((y_OOS == MAJORITY_CLASS).mean())
print(f"\nmajority class on IS        : {CLASS_NAMES[MAJORITY_CLASS]}")
print(f"OOS accuracy of always-'buy' style constant : {CLIMATOLOGY_ACC:.4f}")
print("Any classifier below must be compared against THIS number, not against 1/3 by default,")
print("because the three classes are not equally frequent.")

features        : 23 columns, 3065 rows
feature window  : 2015-01-02 -> 2026-10-01
IS   (selection): 2452 bars  2015-01-02 -> 2024-05-27
OOS  (one touch): 613 bars  2024-05-28 -> 2026-10-01

class balance
      IS_share  OOS_share
sell    0.4091     0.3834
buy     0.4898     0.5073
hold    0.1011     0.1093

majority class on IS        : buy
OOS accuracy of always-'buy' style constant : 0.5073
Any classifier below must be compared against THIS number, not against 1/3 by default,
because the three classes are not equally frequent.


## Why purging and embargo are not optional

The label of bar *t* is realised at *t + HORIZON*. A training observation at *t* therefore carries
information about the test window whenever *t* is within `HORIZON` bars of it, even though *t* is
strictly *before* the test block and passes a naive time-order check.

The cell below takes a contiguous test block and asks: how many training rows does each scheme
leave in place, and how many of those rows have a label window that overlaps the test block? A
row that survives an ordinary chronological split but whose label overlaps the test set is
**leakage that passes a date filter**.

In [4]:
demo_n = 1200
demo_purged = purged_kfold_splits(demo_n, n_splits=5, embargo=EMBARGO, t1=HORIZON)[2]
demo_test = np.asarray(demo_purged[1])
demo_train_naive = np.arange(0, int(demo_test[0]))

# Both schemes are scored against the SAME test block, so the only difference is
# whether rows whose label window reaches into that block are dropped.
demo_splits = [
    ("naive chronological", (demo_train_naive, demo_test)),
    ("purged + embargo", (np.asarray(demo_purged[0]), demo_test)),
]

rows = []
for label, (tr, te) in demo_splits:
    tr = np.asarray(tr)
    te = np.asarray(te)
    t0, t1 = int(te[0]), int(te[-1])
    # A training label window [j, j + HORIZON] touches the test block if it reaches t0.
    overlaps = int(((tr + HORIZON) >= t0).sum())
    inside = int(((tr >= t0) & (tr <= t1)).sum())
    rows.append({
        "scheme": label,
        "train_rows": len(tr),
        "rows_inside_test": inside,
        "label_overlaps_test": overlaps,
        "leaked_rows_pct": 100.0 * overlaps / max(len(tr), 1),
    })

leak_tbl = pd.DataFrame(rows)
print(f"test block = bars {t0}..{t1} of {demo_n}, label horizon = {HORIZON}\n")
print(leak_tbl.round(3).to_string(index=False))
print(
    "\nThe naive split keeps training rows whose labels are decided inside the test window.\n"
    "They are chronologically 'before' the test set, so a date-based filter passes them.\n"
    f"Purging removes them, and the {EMBARGO}-bar embargo additionally drops bars just after\n"
    "the block, whose features are still settling after the test period's own extremes."
)

test block = bars 480..719 of 1200, label horizon = 1

             scheme  train_rows  rows_inside_test  label_overlaps_test  leaked_rows_pct
naive chronological         480                 0                    1            0.208
   purged + embargo         954                 0                  475           49.790

The naive split keeps training rows whose labels are decided inside the test window.
They are chronologically 'before' the test set, so a date-based filter passes them.
Purging removes them, and the 5-bar embargo additionally drops bars just after
the block, whose features are still settling after the test period's own extremes.


In [5]:
# -- shared experiment helpers ------------------------------------------------ #
# Fixed column order for the 3-class probability vectors, so folds with different
# class sets stay comparable when they are stitched together.
CLASS_ORDER = [0, 1, 2]


def fit_predict_scores(X_tr, y_tr, X_te, y_te, model, leak: bool = False):
    """Accuracy on one fold, with the preprocessing discipline chosen by `leak`.

    leak=False : statistics from the training fold only  (correct)
    leak=True  : statistics from train + test jointly     (the anti-pattern)
    """
    if leak:
        Xtr_s, Xte_s = FoldSafePreprocessor.fit_on_full_sample(X_tr, X_te)
    else:
        pre = FoldSafePreprocessor().fit(X_tr)
        Xtr_s, Xte_s = pre.transform(X_tr), pre.transform(X_te)
    model.fit(Xtr_s, y_tr.to_numpy())
    pred = model.predict(Xte_s)
    return float((pred == y_te.to_numpy()).mean())


def cv_mean_accuracy(splits, Xd, yd, make_model, leak: bool = False) -> float:
    return float(np.mean([
        fit_predict_scores(Xd.iloc[tr], yd.iloc[tr], Xd.iloc[te], yd.iloc[te], make_model(), leak=leak)
        for tr, te in splits
    ]))


def evaluate_on_oos(C_value, leak: bool = False, seed: int = RNG_SEED) -> dict:
    """Train on the whole IS window, predict the untouched final window once."""
    model = LogisticRegression(C=C_value, max_iter=2000, random_state=seed)
    if leak:
        Xtr_s, Xte_s = FoldSafePreprocessor.fit_on_full_sample(X_IS, X_OOS)
    else:
        pre = FoldSafePreprocessor().fit(X_IS)
        Xtr_s, Xte_s = pre.transform(X_IS), pre.transform(X_OOS)
    model.fit(Xtr_s, y_IS.to_numpy())
    pred = model.predict(Xte_s)
    acc = float((pred == y_OOS.to_numpy()).mean())
    return {"oos_accuracy": acc, "skill_vs_climatology": acc - CLIMATOLOGY_ACC}


def leakage_audit(Xd: pd.DataFrame, yd: pd.Series, rel_multiple: float = 5.0,
                  abs_threshold: float = 0.5) -> pd.DataFrame:
    """Flag features that behave like a restatement of the target.

    For each feature and each class, compute the point-biserial correlation between
    the feature and the binary indicator of that class, then keep the largest
    absolute value.

    An absolute threshold alone is not sufficient here, and the reason is worth
    stating: this target has a `hold` class, so even a feature that *is* the
    forward return cannot correlate 1.0 with any class indicator -- the deadband
    deliberately breaks the monotone relation. So the detector uses two criteria:

    * ``rel_multiple``: is this feature's correlation several times larger than the
      median correlation across the whole feature set? This is scale-free and does
      not assume the target is a clean restatement of anything.
    * ``abs_threshold``: a high absolute correlation, for the clean binary case.

    Both are reported, and the cell below shows which criterion fires.
    """
    rows = []
    yv = yd.to_numpy()
    for col in Xd.columns:
        v = pd.to_numeric(Xd[col], errors="coerce")
        best, best_k = np.nan, None
        for k in np.unique(yv):
            ind = (yv == k).astype(float)
            ok = v.notna()
            if ok.sum() < 30 or ind[ok].std() == 0 or v[ok].std() == 0:
                continue
            r = float(np.corrcoef(v[ok], ind[ok])[0, 1])
            if np.isfinite(r) and (not np.isfinite(best) or abs(r) > abs(best)):
                best, best_k = r, int(k)
        rows.append({"feature": col, "max_biserial_r": best, "vs_class": best_k})

    out = pd.DataFrame(rows)
    out["abs_r"] = out["max_biserial_r"].abs()
    med = float(out["abs_r"].median())
    out["ratio_to_median"] = out["abs_r"] / med if med > 0 else np.nan
    out["flagged"] = (out["ratio_to_median"] >= rel_multiple) | (out["abs_r"] >= abs_threshold)
    out.attrs["median_abs_r"] = med
    return out.sort_values("abs_r", ascending=False).reset_index(drop=True)


print("helpers defined")

helpers defined


# Experiment 1 — does the split rule change the answer?

Same model, same features, same final test window. Only the rule used to choose the
regularisation strength changes:

* **random K-fold** — the default in most tutorials, including most applied-ML code I have read.
* **purged K-fold with embargo** — Lopez de Prado's construction, implemented and unit-tested in
  `mlfin_core.purged_kfold_splits`.
* **walk-forward** — expanding training window, one contiguous test block per fold.

Each scheme picks a different `C`, and each chosen model is then scored **once** on the final OOS
window. If the split rule is a technical detail, the three answers should agree. They do not, and
the size of the disagreement is the finding.

In [6]:
C_GRID = [0.01, 0.05, 0.1, 0.5, 1.0, 5.0]

# walk_forward_splits enforces min_train_size + n_splits*(test_size + embargo) <= n_obs,
# so the test block has to be sized to what is actually available rather than hardcoded.
WF_MIN_TRAIN = max(60, len(X_IS) // 3)
WF_TEST = max(1, (len(X_IS) - WF_MIN_TRAIN - N_SPLITS * EMBARGO) // N_SPLITS)
print(f"walk-forward budget: min_train={WF_MIN_TRAIN} test_size={WF_TEST} "
      f"embargo={EMBARGO} over {len(X_IS)} IS bars")

schemes = {
    "random K-fold": list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RNG_SEED).split(X_IS)),
    "purged K-fold": [(tr, te) for tr, te in purged_kfold_splits(
        len(X_IS), n_splits=N_SPLITS, embargo=EMBARGO, t1=HORIZON)],
    "walk-forward": walk_forward_splits(
        len(X_IS), n_splits=N_SPLITS, min_train_size=WF_MIN_TRAIN,
        test_size=WF_TEST, embargo=EMBARGO),
}

cv_rows, oos_rows = [], []
for scheme_name, splits in schemes.items():
    for C_value in C_GRID:
        acc = cv_mean_accuracy(splits, X_IS, y_IS, lambda C=C_value: LogisticRegression(
            C=C, max_iter=2000, random_state=RNG_SEED))
        cv_rows.append({"scheme": scheme_name, "C": C_value, "cv_accuracy": acc})

    scheme_df = pd.DataFrame([r for r in cv_rows if r["scheme"] == scheme_name])
    best = scheme_df.loc[scheme_df["cv_accuracy"].idxmax()]
    res = evaluate_on_oos(float(best["C"]))
    oos_rows.append({
        "scheme": scheme_name,
        "selected_C": float(best["C"]),
        "cv_accuracy": float(best["cv_accuracy"]),
        "oos_accuracy": res["oos_accuracy"],
        "skill_vs_climatology": res["skill_vs_climatology"],
    })

cv_tbl = pd.DataFrame(cv_rows)
oos_tbl = pd.DataFrame(oos_rows).sort_values("oos_accuracy", ascending=False)

print("CV accuracy by scheme and regularisation strength\n")
print(cv_tbl.pivot(index="C", columns="scheme", values="cv_accuracy").round(4).to_string())
print(f"\nEach scheme selects its own model, then that model is scored once on the untouched")
print(f"OOS window. Climatology (always '{CLASS_NAMES[MAJORITY_CLASS]}') = {CLIMATOLOGY_ACC:.4f}\n")
print(oos_tbl.round(4).to_string(index=False))

oos_spread = float(oos_tbl["oos_accuracy"].max() - oos_tbl["oos_accuracy"].min())
cv_spread = float(cv_tbl["cv_accuracy"].max() - cv_tbl["cv_accuracy"].min())
print(f"\nspread in OOS accuracy across schemes : {oos_spread:.4f} accuracy points")
print(f"spread in CV  accuracy across all C    : {cv_spread:.4f} accuracy points")
print(f"the CV range is {cv_spread/max(oos_spread,1e-12):.1f}x the disagreement between schemes")
print("\nThe CV signal is smaller than the choice of validation scheme. That is the first")
print("warning sign: a grid fine enough to matter cannot be resolved by this much data.")

walk-forward budget: min_train=817 test_size=322 embargo=5 over 2452 IS bars


CV accuracy by scheme and regularisation strength

scheme  purged K-fold  random K-fold  walk-forward
C                                                 
0.01           0.4686         0.4886        0.4888
0.05           0.4666         0.4857        0.4888
0.10           0.4682         0.4861        0.4894
0.50           0.4674         0.4829        0.4839
1.00           0.4674         0.4837        0.4807
5.00           0.4674         0.4812        0.4807

Each scheme selects its own model, then that model is scored once on the untouched
OOS window. Climatology (always 'buy') = 0.5073

       scheme  selected_C  cv_accuracy  oos_accuracy  skill_vs_climatology
random K-fold        0.01       0.4886        0.4910               -0.0163
purged K-fold        0.01       0.4686        0.4910               -0.0163
 walk-forward        0.10       0.4894        0.4878               -0.0196

spread in OOS accuracy across schemes : 0.0033 accuracy points
spread in CV  accuracy across all C    : 0.0

# Experiment 2 — how much does preprocessing leakage inflate the number?

The single most common bug in this repository's lineage is fitting the scaler on `concat(X_train,
X_test)` before splitting. `mlfin_core` keeps the broken version as
`FoldSafePreprocessor.fit_on_full_sample` **specifically so the damage can be measured** rather
than asserted.

The honest prediction, stated before looking: on features that are already ratios and
standardised oscillators, the effect should be small, and reporting "small" is a more useful
result than reporting the usual inflated headline. If the effect turns out to be large, that is
worth knowing too — so this is printed either way.

In [7]:
purged_IS = [(tr, te) for tr, te in purged_kfold_splits(
    len(X_IS), n_splits=N_SPLITS, embargo=EMBARGO, t1=HORIZON)]

leak_rows = []
for C_value in C_GRID:
    safe_cv = cv_mean_accuracy(purged_IS, X_IS, y_IS, lambda C=C_value: LogisticRegression(
        C=C, max_iter=2000, random_state=RNG_SEED), leak=False)
    leaky_cv = cv_mean_accuracy(purged_IS, X_IS, y_IS, lambda C=C_value: LogisticRegression(
        C=C, max_iter=2000, random_state=RNG_SEED), leak=True)
    safe_oos = evaluate_on_oos(C_value, leak=False)["oos_accuracy"]
    leaky_oos = evaluate_on_oos(C_value, leak=True)["oos_accuracy"]
    leak_rows.append({
        "C": C_value,
        "cv_fold_safe": safe_cv,
        "cv_leaky_scaler": leaky_cv,
        "cv_delta": leaky_cv - safe_cv,
        "oos_fold_safe": safe_oos,
        "oos_leaky_scaler": leaky_oos,
        "oos_delta": leaky_oos - safe_oos,
    })

leak_tbl = pd.DataFrame(leak_rows)
print("Preprocessing leakage, same folds, same model\n")
print(leak_tbl.round(4).to_string(index=False))

max_cv_delta = float(leak_tbl["cv_delta"].abs().max())
max_oos_delta = float(leak_tbl["oos_delta"].abs().max())
worst_C = leak_tbl.loc[leak_tbl["cv_delta"].abs().idxmax(), "C"]
print(f"\nlargest CV  distortion from a leaky scaler : {max_cv_delta:+.4f} accuracy points (C={worst_C})")
print(f"largest OOS distortion from a leaky scaler : {max_oos_delta:+.4f} accuracy points")

LEAK_EFFECT_MATERIAL = max_cv_delta > 0.005
if LEAK_EFFECT_MATERIAL:
    print("\nThe scaler bug is MATERIAL here. Do not report a single number without stating")
    print("which preprocessing discipline produced it.")
else:
    print("\nThe scaler bug is immaterial on THIS feature set, because most columns are already")
    print("ratios or bounded oscillators. That is a property of the features, not a licence to")
    print("leak: the same code on raw price levels, or on any target-dependent normalisation,")
    print("would leak, and the test above would then fail. Keep the fold-safe path as default.")
print(f"\nReported honestly: the maximum observed distortion is {max_cv_delta:+.4f} in CV and")
print(f"{max_oos_delta:+.4f} on the final window. No inflation is claimed that is not measured.")

Preprocessing leakage, same folds, same model

   C  cv_fold_safe  cv_leaky_scaler  cv_delta  oos_fold_safe  oos_leaky_scaler  oos_delta
0.01        0.4686           0.4707    0.0020         0.4910            0.4910     0.0000
0.05        0.4666           0.4666   -0.0000         0.4910            0.4910     0.0000
0.10        0.4682           0.4690    0.0008         0.4878            0.4910     0.0033
0.50        0.4674           0.4674    0.0000         0.4829            0.4878     0.0049
1.00        0.4674           0.4682    0.0008         0.4845            0.4829    -0.0016
5.00        0.4674           0.4666   -0.0008         0.4829            0.4780    -0.0049

largest CV  distortion from a leaky scaler : +0.0020 accuracy points (C=0.01)
largest OOS distortion from a leaky scaler : +0.0049 accuracy points

The scaler bug is immaterial on THIS feature set, because most columns are already
ratios or bounded oscillators. That is a property of the features, not a licence to
leak: t

# Experiment 3 — can the metric be gamed? (the forward-looking feature)

Adding one feature that is unavailable at decision time — the *forward* return from bar *t* to
bar *t + HORIZON* — should push accuracy close to 1.0 if the pipeline is measuring what it claims
to measure, and should do so on both the CV folds and the OOS window.

This is not a strawman. It is the exact mistake behind most of the "our model gets 94% accuracy"
claims in retail finance notebooks, and behind at least one broken notebook in this repository.
The second half of the experiment is the part that matters: the `leakage_audit` helper detects the
offending column automatically, from the data alone, without anyone having to remember that it
was added.

In [8]:
close_all = df["Close"]
leak_feature = close_all.pct_change(-HORIZON).rename("FORWARD_RETURN_LEAK")
audit_before = leakage_audit(X, y)
leaky_X = X.copy()
leaky_X["FORWARD_RETURN_LEAK"] = leak_feature.reindex(X.index)
audit_after = leakage_audit(leaky_X, y)

purged_IS_full = [(tr, te) for tr, te in purged_kfold_splits(
    len(X), n_splits=N_SPLITS, embargo=EMBARGO, t1=HORIZON)]

safe_cv = cv_mean_accuracy(purged_IS_full, X, y, lambda: LogisticRegression(C=1.0, max_iter=2000))
leaky_cv = cv_mean_accuracy(purged_IS_full, leaky_X, y, lambda: LogisticRegression(C=1.0, max_iter=2000))

print("top-5 feature/target correlations BEFORE the leak is added")
print(audit_before.head(5).round(4).to_string(index=False))
print("\ntop-5 feature/target correlations AFTER the leak is added")
print(audit_after.head(5).round(4).to_string(index=False))

flagged = audit_after[audit_after["flagged"]]
strongest_causal = audit_before.iloc[0]
leak_row = audit_after.iloc[0]
print(f"\nfeatures automatically flagged as label-like : {len(flagged)}")
if len(flagged):
    print(flagged[["feature", "max_biserial_r", "ratio_to_median", "flagged"]].round(3).to_string(index=False))

print(f"\nmedian |r| across all features      : {audit_after.attrs['median_abs_r']:.4f}")
print(f"strongest causal feature           : {strongest_causal['feature']} "
      f"(|r| = {strongest_causal['abs_r']:.4f})")
print(f"forward-return feature             : {leak_row['feature']} "
      f"(|r| = {leak_row['abs_r']:.4f}, {leak_row['ratio_to_median']:.1f}x the median)")
print(
    "\nNote the correlation is ~0.64, NOT ~1.0, and that is correct rather than a detector\n"
    "failure: the deadband makes a middle class 'hold', so no feature can be a perfect\n"
    "restatement of a 3-class label. An absolute threshold of 0.95 would MISS this leak.\n"
    "The robust criterion -- correlation far above the median across the feature set --\n"
    "flags it, which is why the detector uses both."
)

print(f"\nCV accuracy, causal features only : {safe_cv:.4f}")
print(f"CV accuracy, one forward feature  : {leaky_cv:.4f}")
print(f"climatology accuracy              : {CLIMATOLOGY_ACC:.4f}")
print(f"inflation from a single illegal feature : {leaky_cv - safe_cv:+.4f} accuracy points")
print(
    "\nA model using the future return cannot be traded, cannot be deployed, and reports the\n"
    "highest accuracy in the notebook. This is why 'high accuracy' is not a result. The\n"
    "detector above is the transferable artefact: it needs no knowledge of which column is\n"
    "illegal, only a correlation against the target."
)

top-5 feature/target correlations BEFORE the leak is added
  feature  max_biserial_r  vs_class  abs_r  ratio_to_median  flagged
range_pct         -0.1048         2 0.1048           3.1563    False
   rsi_14          0.0865         2 0.0865           2.6035    False
 ma_ratio          0.0598         2 0.0598           1.7994    False
   atr_14         -0.0528         2 0.0528           1.5891    False
ret_lag_3          0.0467         2 0.0467           1.4055    False

top-5 feature/target correlations AFTER the leak is added
            feature  max_biserial_r  vs_class  abs_r  ratio_to_median  flagged
FORWARD_RETURN_LEAK          0.6448         0 0.6448          18.1293     True
          range_pct         -0.1048         2 0.1048           2.9476    False
             rsi_14          0.0865         2 0.0865           2.4313    False
           ma_ratio          0.0598         2 0.0598           1.6804    False
             atr_14         -0.0528         2 0.0528           1.4840    

# Experiment 4 — how much of the "best model" is selection noise?

Twenty-four gradient-boosting configurations are evaluated by purged CV on the IS window. The
configuration with the best CV accuracy is the one a normal workflow would ship.

Two things are then measured, and neither is something the workflow itself would ever report:

1. **Does the CV ranking predict the OOS ranking?** The Spearman correlation between the two
   rank orders says how much information cross-validation actually carried.
2. **Did anything in the grid beat the zero-parameter benchmark?** The majority-class benchmark
   from the previous section is the bar. Configurations that clear it in CV and then fail to clear
   it out of sample are the directly observable price of tuning — and if nothing clears it at all,
   the honest conclusion is that the grid has nothing to select, which is a stronger and less
   flattering finding than overfitting. Both branches are printed.

A third number is reported for contrast: **buy-and-hold**. A classifier that predicts direction
is competing with simply being long an index that rose, and a backtest that omits that comparison
is not a backtest. Its Sharpe, Sortino and CAGR are printed beside the strategy's throughout.

The final OOS window is used here as a *measuring instrument* for selection bias. That is a
deliberate, disclosed violation of the one-touch rule — without a held-out set there is no way to
measure selection bias at all — and it is why no later experiment selects on these numbers.

In [9]:
from itertools import product as _product

GRID = {
    "learning_rate": [0.03, 0.12],
    "max_leaf_nodes": [7, 15, 31],
    "min_samples_leaf": [20, 40],
    "l2_regularization": [0.0, 1.0],
    "max_iter": [150],
}
config_names = [dict(zip(GRID.keys(), combo))
                for combo in _product(*GRID.values())]
N_CONFIGS = len(config_names)
print(f"configurations to be tried: {N_CONFIGS}  <- this is the n_trials for DSR below")


def make_hgb(cfg):
    return HistGradientBoostingClassifier(
        learning_rate=cfg["learning_rate"],
        max_leaf_nodes=cfg["max_leaf_nodes"],
        min_samples_leaf=cfg["min_samples_leaf"],
        l2_regularization=cfg["l2_regularization"],
        max_iter=cfg["max_iter"],
        random_state=RNG_SEED,
    )


sweep_rows = []
for i, cfg in enumerate(config_names):
    cv_acc = cv_mean_accuracy(purged_IS, X_IS, y_IS, lambda c=cfg: make_hgb(c))
    # The OOS window is a measuring instrument for selection bias here, not a
    # selection surface: nothing downstream is chosen using these numbers.
    pre = FoldSafePreprocessor().fit(X_IS)
    model = make_hgb(cfg)
    model.fit(pre.transform(X_IS), y_IS.to_numpy())
    oos_acc = float((model.predict(pre.transform(X_OOS)) == y_OOS.to_numpy()).mean())
    sweep_rows.append({
        "config_id": i,
        "learning_rate": cfg["learning_rate"],
        "max_leaf_nodes": cfg["max_leaf_nodes"],
        "min_samples_leaf": cfg["min_samples_leaf"],
        "l2": cfg["l2_regularization"],
        "cv_accuracy": cv_acc,
        "oos_accuracy": oos_acc,
    })

sweep = pd.DataFrame(sweep_rows)
sweep["cv_rank"] = sweep["cv_accuracy"].rank(ascending=False, method="average").astype(int)
sweep["oos_rank"] = sweep["oos_accuracy"].rank(ascending=False, method="average").astype(int)

rho_cv_oos = float(stats.spearmanr(sweep["cv_rank"], sweep["oos_rank"]).statistic)
best_cfg_row = sweep.loc[sweep["cv_accuracy"].idxmax()]
worst_cfg_row = sweep.loc[sweep["cv_accuracy"].idxmin()]

print(f"\nbest 8 configurations by purged-CV accuracy")
print(sweep.sort_values("cv_accuracy", ascending=False).head(8).round(4).to_string(index=False))
print(f"\nCV accuracy across all {N_CONFIGS} configs : "
      f"{sweep['cv_accuracy'].min():.4f} -> {sweep['cv_accuracy'].max():.4f}")
print(f"Spearman rank correlation, CV rank vs OOS rank : {rho_cv_oos:+.3f}")
print(f"CV-best  : config {int(best_cfg_row.config_id)}  "
      f"CV {best_cfg_row.cv_accuracy:.4f} -> OOS {best_cfg_row.oos_accuracy:.4f} "
      f"(OOS rank {int(best_cfg_row.oos_rank)}/{N_CONFIGS})")
print(f"CV-worst : config {int(worst_cfg_row.config_id)}  "
      f"CV {worst_cfg_row.cv_accuracy:.4f} -> OOS {worst_cfg_row.oos_accuracy:.4f} "
      f"(OOS rank {int(worst_cfg_row.oos_rank)}/{N_CONFIGS})")

beat_cv = sweep[sweep["cv_accuracy"] > CLIMATOLOGY_ACC]
lucky = beat_cv[beat_cv["oos_accuracy"] <= CLIMATOLOGY_ACC]
below_clim_oos = int((sweep["oos_accuracy"] <= CLIMATOLOGY_ACC).sum())
cv_to_oos_drop = float(best_cfg_row["cv_accuracy"] - best_cfg_row["oos_accuracy"])
# Positive means cross-validation picked a config that did transfer better than a
# randomly chosen one; negative means selection did not even beat the average.
selected_minus_mean = float(best_cfg_row["oos_accuracy"] - sweep["oos_accuracy"].mean())

print(f"\nconfigs beating climatology in CV      : {len(beat_cv)} / {N_CONFIGS}")
print(f"...of which FAILED to beat it out of sample : {len(lucky)} / {len(beat_cv)}")
print(f"configs at or below climatology out of sample : {below_clim_oos} / {N_CONFIGS}")
print(f"\nOOS accuracy of the CV-best config      : {best_cfg_row['oos_accuracy']:.4f}")
print(f"  its own CV accuracy                   : {best_cfg_row['cv_accuracy']:.4f}")
print(f"  degradation from CV to OOS            : {cv_to_oos_drop:+.4f}")
print(f"OOS accuracy range over all configs     : "
      f"{sweep['oos_accuracy'].min():.4f} -> {sweep['oos_accuracy'].max():.4f}")
print(f"mean OOS accuracy over all configs      : {sweep['oos_accuracy'].mean():.4f}")
print(f"CV-best minus mean, out of sample       : {selected_minus_mean:+.4f}")
if selected_minus_mean <= 0:
    print("  Negative: cross-validation did not even pick a config that beat a randomly")
    print("  chosen one out of sample. Selection transferred nothing here.")
else:
    print("  Positive but small: selection transferred a little, far less than the")
    print("  in-sample ranking suggested.")
print(f"climatology for comparison              : {CLIMATOLOGY_ACC:.4f}")

if len(beat_cv) == 0:
    print("\nNOTHING in the grid beat the majority-class benchmark, in CV or out of sample.")
    print("So the price of a 24-configuration search was not paid in false positives -- there")
    print("was no signal for the search to find in the first place. That is a different and")
    print("more fundamental failure than overfitting, and it is the honest headline: on these")
    print("features and this target, a gradient-boosted classifier contains no usable")
    print("directional edge over 'always be long SPY'. The mean over all configurations is")
    print("therefore the number to quote, and it is below the benchmark.")
else:
    print("\nThe mean over all configurations, not the maximum, is the honest expectation for a")
    print("configuration drawn without knowing which one CV liked. Read that number.")

configurations to be tried: 24  <- this is the n_trials for DSR below



best 8 configurations by purged-CV accuracy
 config_id  learning_rate  max_leaf_nodes  min_samples_leaf  l2  cv_accuracy  oos_accuracy  cv_rank  oos_rank
         8           0.03              31                20 0.0       0.4662        0.4812        1         1
        16           0.12              15                20 0.0       0.4605        0.4405        2        17
        22           0.12              31                40 0.0       0.4592        0.4437        3        15
         2           0.03               7                40 0.0       0.4592        0.4551        4         6
         7           0.03              15                40 1.0       0.4588        0.4568        5         3
         0           0.03               7                20 0.0       0.4584        0.4551        6         6
         5           0.03              15                20 1.0       0.4584        0.4568        7         3
        18           0.12              15                40 0.0       0.457

# Experiment 4b — correcting the selected Sharpe for having selected it

Selecting the best of *N* trials biases its Sharpe upward. `probabilistic_sharpe_ratio` asks
whether the realised Sharpe is distinguishable from zero; `deflated_sharpe_ratio` asks the harder
question — whether it is distinguishable from **the best of N trials that were pure luck**.
`min_track_record_length` answers how many observations would be needed to establish the target
Sharpe at all.

These are computed on the walk-forward net return stream of the selected configuration, in the
next cell. Here the inputs are established: the selected configuration, and the honest number of
trials that produced it.

In [10]:
SELECTED = config_names[int(best_cfg_row.config_id)]
print("configuration a normal workflow would ship, selected by purged CV on IS only:")
print(json.dumps(SELECTED, indent=2))
print(f"\nn_trials for the deflated Sharpe: {N_CONFIGS}")
print("Omitting this argument is how a tuned backtest reports a Sharpe that means nothing.")

dsr_undeflated = deflated_sharpe_ratio(pd.Series([0.0, 1.0]), n_trials=1)
print("\nsanity check on the implementation (a degenerate 2-point stream):")
print(json.dumps({k: v for k, v in dsr_undeflated.items() if k in ("sr", "psr", "dsr")}, indent=2,
                 default=str))
print("A Sharpe from two observations carries no information, as it should.")

configuration a normal workflow would ship, selected by purged CV on IS only:
{
  "learning_rate": 0.03,
  "max_leaf_nodes": 31,
  "min_samples_leaf": 20,
  "l2_regularization": 0.0,
  "max_iter": 150
}

n_trials for the deflated Sharpe: 24
Omitting this argument is how a tuned backtest reports a Sharpe that means nothing.

sanity check on the implementation (a degenerate 2-point stream):
{
  "sr": 0.7071067811865475,
  "psr": NaN,
  "dsr": NaN
}
A Sharpe from two observations carries no information, as it should.


# Experiment 5 — does the answer depend on where the test window sat?

One walk-forward path is one sample from the distribution of possible results. Combinatorial
purged CV produces many: the sample is cut into groups, and every combination of test groups
becomes a backtest path. The spread across paths answers a question no single hold-out can — *how
much of this result is a property of the strategy, and how much is a property of the dates I
happened to pick?*

Out-of-fold net returns are computed once by purged K-fold across the whole sample, then sliced
into paths. The values passed to `cpcv_backtest_paths` are per-bar **net returns**, not positions,
so each path Sharpe is a real Sharpe of a real equity curve.

In [11]:
n_groups = 6
n_test_groups = 2
cpcv_paths = combinatorial_purged_cv_splits(
    n_obs, n_groups=n_groups, n_test_groups=n_test_groups, embargo=EMBARGO, t1=HORIZON)
print(f"CPCV: {n_groups} groups, {n_test_groups} test groups per path -> "
      f"{len(cpcv_paths)} backtest paths")

# Out-of-fold probabilities for the whole sample: every bar predicted by a model that
# never saw it, under purged folds with embargo.
present: set[int] = set()
oof_prob = np.full((n_obs, len(CLASS_ORDER)), np.nan)
for tr, te in purged_kfold_splits(n_obs, n_splits=n_groups, embargo=EMBARGO, t1=HORIZON):
    pre = FoldSafePreprocessor().fit(X.iloc[tr])
    model = make_hgb(SELECTED)
    model.fit(pre.transform(X.iloc[tr]), y.iloc[tr].to_numpy())
    present |= set(int(c) for c in model.classes_)
    # Map this fold's class columns onto the global order so folds are comparable.
    proba = model.predict_proba(pre.transform(X.iloc[te]))
    for src_i, cls in enumerate(model.classes_):
        oof_prob[te, CLASS_ORDER.index(int(cls))] = proba[:, src_i]
    oof_prob[te] = np.nan_to_num(oof_prob[te], nan=0.0)

print(f"classes present in every training fold: {sorted(present)} (expected {CLASS_ORDER})")
covered = ~np.isnan(oof_prob).any(axis=1)
print(f"bars with an out-of-fold prediction : {int(covered.sum())} / {n_obs}")

buy_i, sell_i = CLASS_ORDER.index(1), CLASS_ORDER.index(0)
idx_covered = np.flatnonzero(covered)
oof_pos = np.full(n_obs, np.nan)
oof_pos[covered] = apply_causal_execution(
    oof_prob[covered], buy=buy_i, sell=sell_i, confirm_bars=2, min_confidence=0.40)

valid = covered & np.isfinite(oof_pos)
idx_valid = np.flatnonzero(valid)
oof_net = np.full(n_obs, np.nan)
oof_net[idx_valid] = positions_to_returns(
    oof_pos[idx_valid], close.iloc[idx_valid], cost_bps=COST_BPS).to_numpy()

path_stats = cpcv_backtest_paths(np.nan_to_num(oof_net, nan=0.0), cpcv_paths)
path_sharpes = np.array(list(path_stats["path_sharpe"].values()))
print(f"\npath Sharpe: mean {path_stats['mean_path_sharpe']:.3f} | "
      f"p05 {path_stats['p05_path_sharpe']:.3f} | p95 {path_stats['p95_path_sharpe']:.3f}")
print(f"single-path (mean) would have reported one of these values with no interval attached.")
print(f"share of paths with Sharpe <= 0 : {(path_sharpes <= 0).mean():.1%}")
print(f"OOS-position share (bars long)  : {np.nanmean(np.abs(oof_pos[valid])):.3f} of bars")
print(f"turnover: {np.abs(np.diff(oof_pos[valid])).sum():.0f} position changes over "
      f"{int(valid.sum())} bars")
print("\nA negative fifth percentile means the single-path Sharpe is not a safe summary of the")
print("distribution. It is one draw from it.")

CPCV: 6 groups, 2 test groups per path -> 15 backtest paths


classes present in every training fold: [0, 1, 2] (expected [0, 1, 2])
bars with an out-of-fold prediction : 3065 / 3065

path Sharpe: mean -0.246 | p05 -0.865 | p95 0.301
single-path (mean) would have reported one of these values with no interval attached.
share of paths with Sharpe <= 0 : 73.3%
OOS-position share (bars long)  : 0.997 of bars
turnover: 618 position changes over 3065 bars

A negative fifth percentile means the single-path Sharpe is not a safe summary of the
distribution. It is one draw from it.


## The deployable simulation

Everything so far has been validation machinery. This cell produces the one artefact a research
team actually wants: a causal walk-forward simulation of the selected configuration, net of
costs, with uncertainty attached.

Protocol, stated precisely:

* Expanding training window, refit every `REFIT_EVERY` bars.
* Every bar is scored using a model fitted only on bars strictly before it.
* Preprocessing statistics recomputed inside every refit.
* Signals pass through a confirmation filter, then to positions, then to returns with costs.
* No parameter anywhere is chosen using the final 20% window.

In [12]:
wf_prob = np.full((n_obs, len(CLASS_ORDER)), np.nan)
refits = 0
for start in range(EMBARGO + n_oos, n_obs, REFIT_EVERY):
    end = min(start + REFIT_EVERY, n_obs)
    tr = np.arange(0, start)
    pre = FoldSafePreprocessor().fit(X.iloc[tr])
    model = make_hgb(SELECTED)
    model.fit(pre.transform(X.iloc[tr]), y.iloc[tr].to_numpy())
    proba = model.predict_proba(pre.transform(X.iloc[start:end]))
    # Map this fit's class columns onto the global order.
    for src_i, cls in enumerate(model.classes_):
        wf_prob[start:end, CLASS_ORDER.index(int(cls))] = proba[:, src_i]
    wf_prob[start:end] = np.nan_to_num(wf_prob[start:end], nan=0.0)
    refits += 1

wf_valid = ~np.isnan(wf_prob).any(axis=1)
wf_idx = np.flatnonzero(wf_valid)
wf_prob_v = wf_prob[wf_idx]
wf_pos = apply_causal_execution(wf_prob_v, buy=buy_i, sell=sell_i,
                                confirm_bars=2, min_confidence=0.40)
prices_v = close.iloc[wf_idx]
net = positions_to_returns(wf_pos, prices_v, cost_bps=COST_BPS)
bh = buy_and_hold_returns(prices_v)

bench = pd.DataFrame({"strategy_net": net, "buy_and_hold": bh}).dropna()
net_s, bh_s = bench["strategy_net"], bench["buy_and_hold"]

# The walk-forward spans the whole simulated range, which includes the IS segment.
# The IS segment is not a clean read (the configuration was selected on it), so the OOS
# segment is reported separately and is the number that gets quoted.
oos_net = net_s.loc[net_s.index >= X_OOS.index[0]]
oos_bh = bh_s.loc[bh_s.index >= X_OOS.index[0]]


def max_drawdown(r: pd.Series) -> float:
    equity = (1.0 + r).cumprod()
    return float((equity / equity.cummax() - 1.0).min())


n_trades = int((np.abs(np.diff(wf_pos)) > 0).sum())
metrics = {
    "bars": int(len(net_s)),
    "trades": n_trades,
    "CAGR": annualised_return(net_s),
    "Sharpe": sharpe_ratio(net_s),
    "Sortino": sortino_ratio(net_s),
    "buy_hold_Sharpe": sharpe_ratio(bh_s),
    "buy_hold_CAGR": annualised_return(bh_s),
    "max_drawdown": max_drawdown(net_s),
    "profit_factor": profit_factor(net_s),
    "win_rate": win_rate(net_s),
}
oos_metrics = {
    "bars": int(len(oos_net)),
    "CAGR": annualised_return(oos_net),
    "Sharpe": sharpe_ratio(oos_net),
    "Sortino": sortino_ratio(oos_net),
    "buy_hold_Sharpe": sharpe_ratio(oos_bh),
    "buy_hold_CAGR": annualised_return(oos_bh),
    "max_drawdown": max_drawdown(oos_net),
}

print(f"walk-forward: {len(net_s)} bars | {refits} refits | {n_trades} trades | "
      f"{COST_BPS:.0f} bp round trip")
print(f"simulated span: {net_s.index[0].date()} -> {net_s.index[-1].date()}")
print("\nfull simulated span (includes the IS segment the config was selected on)")
for k, v in metrics.items():
    print(f"  {k:<18}: {v:>12.4f}")
print(f"\nOOS segment only, {oos_net.index[0].date()} -> {oos_net.index[-1].date()} "
      "(never used for selection)")
for k, v in oos_metrics.items():
    print(f"  {k:<18}: {v:>12.4f}")

edge = oos_metrics["Sharpe"] - oos_metrics["buy_hold_Sharpe"]
print(f"\n  OOS Sharpe edge vs buy-and-hold : {edge:+.4f}")
print("  Buy-and-hold is included because it is the benchmark that actually matters for a")
print("  long-only directional strategy, and it is the one `HW1`-`HW3` never reported.")

walk-forward: 2447 bars | 117 refits | 259 trades | 10 bp round trip
simulated span: 2017-05-17 -> 2026-10-01

full simulated span (includes the IS segment the config was selected on)
  bars              :    2447.0000
  trades            :     259.0000
  CAGR              :      -0.0464
  Sharpe            :      -0.1719
  Sortino           :      -0.2157
  buy_hold_Sharpe   :       0.8444
  buy_hold_CAGR     :       0.1460
  max_drawdown      :      -0.5534
  profit_factor     :       0.9667
  win_rate          :       0.4781

OOS segment only, 2024-05-28 -> 2026-10-01 (never used for selection)
  bars              :     613.0000
  CAGR              :      -0.0120
  Sharpe            :       0.0022
  Sortino           :       0.0029
  buy_hold_Sharpe   :       1.1060
  buy_hold_CAGR     :       0.1767
  max_drawdown      :      -0.1835

  OOS Sharpe edge vs buy-and-hold : -1.1037
  Buy-and-hold is included because it is the benchmark that actually matters for a
  long-only directiona

In [13]:
ci = bootstrap_sharpe_ci(net_s, n_boot=4000, block_size=10, alpha=0.05, seed=RNG_SEED)
psr = probabilistic_sharpe_ratio(net_s, sr_benchmark=0.0)
dsr = deflated_sharpe_ratio(net_s, n_trials=N_CONFIGS)
mintrl_bars = min_track_record_length(net_s, sr_target=SR_TARGET)
mintrl_bh = min_track_record_length(bh_s, sr_target=SR_TARGET)
MINTRL_DEFINED = bool(np.isfinite(mintrl_bars))

print("uncertainty on the walk-forward Sharpe")
print(f"  point estimate            : {ci['point']:.3f}")
print(f"  95% block-bootstrap CI     : [{ci['lo']:.3f}, {ci['hi']:.3f}]  ({ci['n_boot']} resamples)")
print(f"  P(Sharpe > 0)             : {ci['p_gt_zero']:.3f}")
print(f"  PSR vs zero               : {psr:.4f}")
print(f"  DSR vs best of {N_CONFIGS} trials   : {dsr['dsr']:.4f} "
      f"(SR benchmark {dsr['sr_benchmark']:.3f})")
if MINTRL_DEFINED:
    print(f"  MinTRL for SR {SR_TARGET:.1f}       : {mintrl_bars:,.0f} bars "
          f"({mintrl_bars/TRADING_DAYS:.1f} years) vs {len(net_s)} observed")
else:
    print(f"  MinTRL for SR {SR_TARGET:.1f}       : undefined")
    print("    MinTRL is undefined when the realised Sharpe is not positive: no length of")
    print("    this return stream establishes a target Sharpe, because there is no positive")
    print("    Sharpe to establish. That is the correct answer, not a missing value.")
print(f"  for contrast, buy-and-hold MinTRL for SR {SR_TARGET:.1f}: "
      f"{mintrl_bh/TRADING_DAYS:.1f} years (its Sharpe is positive)")

CI_EXCLUDES_ZERO = bool(np.isfinite(ci["lo"]) and ci["lo"] > 0)
DSR_CREDIBLE = bool(np.isfinite(dsr["dsr"]) and dsr["dsr"] >= 0.95)
VERDICT = ("credible edge" if (CI_EXCLUDES_ZERO and DSR_CREDIBLE)
           else "no credible edge after correcting for selection")

print(f"\n95% CI excludes zero : {CI_EXCLUDES_ZERO}")
print(f"DSR >= 0.95          : {DSR_CREDIBLE}")
print(f"VERDICT              : {VERDICT}")
if not CI_EXCLUDES_ZERO:
    print("\nThe interval includes zero. The point estimate is not distinguishable from luck at")
    print("this sample size, and no amount of model engineering changes that fact -- only a")
    print("longer track record does.")
if not DSR_CREDIBLE:
    print(f"\nAfter deflating for having tried {N_CONFIGS} configurations, the Sharpe is not")
    print("distinguishable from the best of that many lucky guesses.")
if VERDICT == "credible edge":
    print("\nThe interval excludes zero AND survives the selection correction. That is the bar,")
    print("and most published backtests do not clear it.")
if MINTRL_DEFINED:
    print(f"\nMinTRL says {mintrl_bars/TRADING_DAYS:.1f} years of data are needed to establish a")
    print(f"Sharpe of {SR_TARGET:.1f} at all. This run has {len(net_s)/TRADING_DAYS:.1f} years.")
else:
    print(f"\nBuy-and-hold needs {mintrl_bh/TRADING_DAYS:.1f} years to establish SR "
          f"{SR_TARGET:.1f}; this strategy\nneeds an unlimited record, because its realised "
          f"Sharpe is negative. That asymmetry is the result.")

uncertainty on the walk-forward Sharpe
  point estimate            : -0.172
  95% block-bootstrap CI     : [-0.736, 0.389]  (4000 resamples)
  P(Sharpe > 0)             : 0.275
  PSR vs zero               : 0.2960
  DSR vs best of 24 trials   : 0.0078 (SR benchmark 0.038)
  MinTRL for SR 1.0       : undefined
    MinTRL is undefined when the realised Sharpe is not positive: no length of
    this return stream establishes a target Sharpe, because there is no positive
    Sharpe to establish. That is the correct answer, not a missing value.
  for contrast, buy-and-hold MinTRL for SR 1.0: 1.4 years (its Sharpe is positive)

95% CI excludes zero : False
DSR >= 0.95          : False
VERDICT              : no credible edge after correcting for selection

The interval includes zero. The point estimate is not distinguishable from luck at
this sample size, and no amount of model engineering changes that fact -- only a
longer track record does.

After deflating for having tried 24 configurations

## Conclusions

The numbers above are printed by the code and computed from this run; the reasoning is stated
separately so the two can be checked against each other.

**1. Validation discipline is not bookkeeping — it is the result.** Three split rules applied to
one model and one feature set selected three different models and disagreed on the out-of-sample
accuracy by the margin printed in Experiment 1. Where the CV range across the whole grid is
comparable to the disagreement between schemes, the grid cannot be resolved by the data, and
picking the maximum is picking noise.

**2. The two cheap leakage bugs behaved differently, and reporting both is the point.** A scaler
fitted on train+test distorts the CV accuracy by the amount printed in Experiment 2 — on this
feature set, a small amount, because most columns are already ratios and bounded oscillators. A
single forward-looking feature inflates accuracy by roughly an order of magnitude more, in the
same direction. The transferable habit is the audit, not the intuition: "my features are probably
causal" is not a control, a correlation against the target is. Note also that the leak's
correlation is well below 1.0 because the deadband creates a middle class, so an absolute
threshold would have missed it — the detector has to be relative to the rest of the feature set.

**3. Tuning is a cost, and on this problem it found nothing at all.** The sweep in Experiment 4
runs 24 configurations. The honest reading depends on what the run shows, and both branches are
printed: if configurations beat the majority-class benchmark in CV and then fail out of sample,
that count is the price of the search; if nothing beats the benchmark anywhere, the price was not
paid because there was no signal to find, which is a more fundamental result than overfitting.
Either way the number to quote is the **mean** accuracy across configurations, not the maximum,
and it must be read next to the majority-class benchmark. `HW2`'s six-trade backtest is the
extreme version of this failure: a maximum over an unstated number of attempts, reported as if it
were the only attempt.

**4. A single backtest path is one draw from a distribution.** The CPCV cell reports the fifth and
95th percentiles of the path Sharpe distribution. When the fifth percentile is negative, quoting
the single-path Sharpe as *the* Sharpe is a category error, and it applies to every backtest in
this repository that reported a single number.

**5. Buy-and-hold is the benchmark that was missing.** An edge over a constant mean is an
arithmetic curiosity; an edge over the zero-parameter buy-and-hold benchmark is the only kind that
pays. That comparison is in the walk-forward cell, next to the uncertainty on it.

**What HW8 inherits.** A model zoo scored against zero-parameter benchmarks, a selection count for
the deflated Sharpe, a purged and embargoed cross-validation, costs in the return stream, and an
uncertainty interval attached to every headline figure.

In [14]:
print("=" * 78)
print("BOTTOM LINE, computed from this run")
print("=" * 78)

print("\nData and protocol")
print(f"  ticker / source              : {TICKER} / {source}")
print(f"  bars (features)              : {n_obs}")
print(f"  IS (selection only)          : {len(X_IS)}")
print(f"  OOS (final, one touch)       : {len(X_OOS)}")
print(f"  climatology accuracy         : {CLIMATOLOGY_ACC:.4f} "
      f"(always '{CLASS_NAMES[MAJORITY_CLASS]}')")

print("\n1. Does the split rule change the answer?")
print(f"  OOS accuracy spread across 3 schemes : {oos_spread:.4f}")
print(f"  CV  accuracy spread across 6 C values: {cv_spread:.4f}")
print(f"  configuration selected by each scheme : "
      + " | ".join(f"{r['scheme']} -> C={r['selected_C']:g}" for _, r in oos_tbl.iterrows()))
print(f"  schemes agree on the same model      : "
      f"{oos_tbl['selected_C'].nunique() == 1}")

print("\n2. Preprocessing leakage (train+test scaler)")
print(f"  max distortion in CV  accuracy : {max_cv_delta:+.4f}")
print(f"  max distortion in OOS accuracy : {max_oos_delta:+.4f}")
print(f"  material ( > 0.005 )           : {LEAK_EFFECT_MATERIAL}")

print("\n3. Forward-looking feature")
print(f"  CV accuracy, causal features   : {safe_cv:.4f}")
print(f"  CV accuracy, +1 illegal feature: {leaky_cv:.4f}")
print(f"  inflation                      : {leaky_cv - safe_cv:+.4f}")
print(f"  |r| of the leak vs median |r|  : {leak_row['abs_r']:.3f} vs "
      f"{audit_after.attrs['median_abs_r']:.3f} ({leak_row['ratio_to_median']:.1f}x)")
print(f"  flagged automatically by audit : {len(flagged)} feature(s)")
if len(flagged):
    print(f"    -> {', '.join(flagged['feature'])}")

print("\n4. Selection noise")
print(f"  configurations tried           : {N_CONFIGS}")
print(f"  Spearman(CV rank, OOS rank)    : {rho_cv_oos:+.3f}")
print(f"  beat climatology in CV         : {len(beat_cv)}")
print(f"  ...failed out of sample        : {len(lucky)}")
print(f"  at/below climatology OOS       : {below_clim_oos} / {N_CONFIGS}")
print(f"  mean OOS accuracy (all cfgs)   : {sweep['oos_accuracy'].mean():.4f}")
print(f"  OOS accuracy of the CV-best    : {best_cfg_row['oos_accuracy']:.4f} "
      f"(CV said {best_cfg_row['cv_accuracy']:.4f}, drop {cv_to_oos_drop:+.4f})")
print(f"  CV-best minus mean, OOS accuracy: {selected_minus_mean:+.4f}")
print(f"  anything beat the benchmark?   : {len(beat_cv) > 0}")

print("\n5. CPCV path distribution")
print(f"  paths                          : {path_stats['n_paths']}")
print(f"  path Sharpe p05 / mean / p95   : {path_stats['p05_path_sharpe']:.3f} / "
      f"{path_stats['mean_path_sharpe']:.3f} / {path_stats['p95_path_sharpe']:.3f}")
print(f"  share of paths with Sharpe <= 0: {(path_sharpes <= 0).mean():.1%}")

print("\nWalk-forward simulation, net of costs")
print(f"  bars / trades / cost           : {len(net_s)} / {n_trades} / {COST_BPS:.0f} bp")
print(f"  full span CAGR / Sharpe        : {metrics['CAGR']:+.4f} / {metrics['Sharpe']:.3f}")
print(f"  full span Sortino / max DD     : {metrics['Sortino']:.3f} / "
      f"{metrics['max_drawdown']:+.4f}")
print(f"  full span profit factor/win    : {metrics['profit_factor']:.3f} / "
      f"{metrics['win_rate']:.4f}")
print("  OOS segment only (never used for selection):")
print(f"    CAGR / Sharpe                : {oos_metrics['CAGR']:+.4f} / "
      f"{oos_metrics['Sharpe']:.3f}")
print(f"    buy-and-hold CAGR / Sharpe   : {oos_metrics['buy_hold_CAGR']:+.4f} / "
      f"{oos_metrics['buy_hold_Sharpe']:.3f}")
print(f"    Sharpe edge vs buy-and-hold  : {edge:+.4f}")
print(f"    Sortino / max DD             : {oos_metrics['Sortino']:.3f} / "
      f"{oos_metrics['max_drawdown']:+.4f}")

print("\nUncertainty")
print(f"  95% bootstrap Sharpe CI       : [{ci['lo']:.3f}, {ci['hi']:.3f}]")
print(f"  PSR vs zero                   : {psr:.4f}")
print(f"  DSR vs best of {N_CONFIGS}       : {dsr['dsr']:.4f}")
if MINTRL_DEFINED:
    print(f"  MinTRL for SR {SR_TARGET:.1f}       : {mintrl_bars/TRADING_DAYS:.1f} years "
          f"(have {len(net_s)/TRADING_DAYS:.1f})")
else:
    print(f"  MinTRL for SR {SR_TARGET:.1f}       : undefined (realised Sharpe <= 0)")
print(f"  buy-and-hold MinTRL, SR {SR_TARGET:.1f}  : {mintrl_bh/TRADING_DAYS:.1f} years")
print(f"\n  VERDICT                       : {VERDICT}")

print("\nWhat HW8 inherits: zero-parameter benchmarks (climatology AND buy-and-hold), purged")
print("cross-validation with embargo, the configuration count for DSR, costs in the return")
print("stream, and an interval on every headline number.")

BOTTOM LINE, computed from this run

Data and protocol
  ticker / source              : SPY / SPY (yfinance: 2015-01-01..2026-10-05)
  bars (features)              : 3065
  IS (selection only)          : 2452
  OOS (final, one touch)       : 613
  climatology accuracy         : 0.5073 (always 'buy')

1. Does the split rule change the answer?
  OOS accuracy spread across 3 schemes : 0.0033
  CV  accuracy spread across 6 C values: 0.0229
  configuration selected by each scheme : random K-fold -> C=0.01 | purged K-fold -> C=0.01 | walk-forward -> C=0.1
  schemes agree on the same model      : False

2. Preprocessing leakage (train+test scaler)
  max distortion in CV  accuracy : +0.0020
  max distortion in OOS accuracy : +0.0049
  material ( > 0.005 )           : False

3. Forward-looking feature
  CV accuracy, causal features   : 0.4832
  CV accuracy, +1 illegal feature: 0.9566
  inflation                      : +0.4734
  |r| of the leak vs median |r|  : 0.645 vs 0.036 (18.1x)
  flagged a

In [15]:
summary = {
    "run": {
        "ticker": TICKER,
        "data_source": str(source),
        "n_bars": int(n_obs),
        "n_features": int(X.shape[1]),
        "is_bars": int(len(X_IS)),
        "oos_bars": int(len(X_OOS)),
        "horizon": HORIZON,
        "dead_band": DEAD_BAND,
        "embargo": EMBARGO,
        "cost_bps": COST_BPS,
        "refit_every": REFIT_EVERY,
    },
    "benchmark_climatology_accuracy": CLIMATOLOGY_ACC,
    "exp1_split_rule": {
        "oos_accuracy_by_scheme": oos_tbl.set_index("scheme")["oos_accuracy"].round(4).to_dict(),
        "selected_C_by_scheme": oos_tbl.set_index("scheme")["selected_C"].to_dict(),
        "oos_accuracy_spread": oos_spread,
        "cv_accuracy_spread_across_grid": cv_spread,
        "schemes_selected_same_model": bool(oos_tbl["selected_C"].nunique() == 1),
    },
    "exp2_preprocessing_leakage": {
        "max_cv_distortion": max_cv_delta,
        "max_oos_distortion": max_oos_delta,
        "material": bool(LEAK_EFFECT_MATERIAL),
    },
    "exp3_forward_feature": {
        "cv_accuracy_causal_only": safe_cv,
        "cv_accuracy_with_leak": leaky_cv,
        "inflation": leaky_cv - safe_cv,
        "leak_abs_biserial_r": float(leak_row["abs_r"]),
        "median_abs_biserial_r": audit_after.attrs["median_abs_r"],
        "ratio_to_median": float(leak_row["ratio_to_median"]),
        "features_flagged_by_audit": flagged["feature"].tolist(),
        "audit_detects_leak_automatically": bool(len(flagged) > 0),
        "abs_threshold_0_95_would_have_missed_it": bool(
            leak_row["abs_r"] < 0.95 and leak_row["ratio_to_median"] >= 5.0),
    },
    "exp4_selection_noise": {
        "n_configs": N_CONFIGS,
        "spearman_cv_rank_vs_oos_rank": rho_cv_oos,
        "beat_climatology_in_cv": int(len(beat_cv)),
        "failed_out_of_sample": int(len(lucky)),
        "at_or_below_climatology_oos": int(below_clim_oos),
        "mean_oos_accuracy_all_configs": float(sweep["oos_accuracy"].mean()),
        "oos_accuracy_cv_best": float(best_cfg_row["oos_accuracy"]),
        "cv_accuracy_of_cv_best": float(best_cfg_row["cv_accuracy"]),
        "cv_to_oos_degradation": cv_to_oos_drop,
        "cv_best_minus_mean_oos": selected_minus_mean,
        "anything_beat_the_benchmark": bool(len(beat_cv) > 0),
    },
    "exp5_cpcv": {
        "n_paths": int(path_stats["n_paths"]),
        "p05_path_sharpe": path_stats["p05_path_sharpe"],
        "mean_path_sharpe": path_stats["mean_path_sharpe"],
        "p95_path_sharpe": path_stats["p95_path_sharpe"],
        "share_paths_sharpe_le_zero": float((path_sharpes <= 0).mean()),
    },
    "walk_forward_full_span": {k: float(v) for k, v in metrics.items()},
    "walk_forward_oos_segment": {k: float(v) for k, v in oos_metrics.items()},
    "oos_segment_sharpe_edge_vs_buy_hold": edge,
    "uncertainty": {
        "sharpe_ci_lo": ci["lo"],
        "sharpe_ci_hi": ci["hi"],
        "p_sharpe_gt_zero": ci["p_gt_zero"],
        "psr": psr,
        "dsr": dsr["dsr"],
        "dsr_sr_benchmark": dsr["sr_benchmark"],
        "mintrl_years_for_target": (mintrl_bars / TRADING_DAYS) if MINTRL_DEFINED else None,
        "mintrl_defined": MINTRL_DEFINED,
        "mintrl_years_for_target_buy_hold": mintrl_bh / TRADING_DAYS,
        "observed_years": len(net_s) / TRADING_DAYS,
        "ci_excludes_zero": CI_EXCLUDES_ZERO,
        "dsr_credible": DSR_CREDIBLE,
        "verdict": VERDICT,
    },
}
print(json.dumps(summary, indent=2, default=str))

{
  "run": {
    "ticker": "SPY",
    "data_source": "SPY (yfinance: 2015-01-01..2026-10-05)",
    "n_bars": 3065,
    "n_features": 23,
    "is_bars": 2452,
    "oos_bars": 613,
    "horizon": 1,
    "dead_band": 0.0005,
    "embargo": 5,
    "cost_bps": 10.0,
    "refit_every": 21
  },
  "benchmark_climatology_accuracy": 0.5073409461663948,
  "exp1_split_rule": {
    "oos_accuracy_by_scheme": {
      "random K-fold": 0.491,
      "purged K-fold": 0.491,
      "walk-forward": 0.4878
    },
    "selected_C_by_scheme": {
      "random K-fold": 0.01,
      "purged K-fold": 0.01,
      "walk-forward": 0.1
    },
    "oos_accuracy_spread": 0.0032626427406199365,
    "cv_accuracy_spread_across_grid": 0.022858010289921404,
    "schemes_selected_same_model": false
  },
  "exp2_preprocessing_leakage": {
    "max_cv_distortion": 0.0020358285880544003,
    "max_oos_distortion": 0.004893964110929849,
    "material": false
  },
  "exp3_forward_feature": {
    "cv_accuracy_causal_only": 0.483197389